In [1]:
import pandas as pd
import re

In [2]:
train = pd.read_csv("train_data.csv")
test = pd.read_csv("test_data.csv")

train.head()

,word_id,word,answer,participant_id,text
0,enc_wikimoon_13_page_1_0,Luna,0,28,enc_wikimoon
1,enc_wikimoon_13_page_1_1,https://ro.wikipedia.org/wiki/Luna,668,28,enc_wikimoon
2,enc_wikimoon_13_page_1_2,De,0,28,enc_wikimoon
3,enc_wikimoon_13_page_1_3,la,220,28,enc_wikimoon
4,enc_wikimoon_13_page_1_4,"Wikipedia,",0,28,enc_wikimoon


In [3]:
def clean(text):
    text = text.lower()
    return text

cat_cols = train.select_dtypes(include = "object").columns

for col in cat_cols:
    train[col] = train[col].apply(clean)
    test[col] = test[col].apply(clean)

In [4]:
# 3 am feature engineering :)
X = train.drop(columns = ["word_id", "answer", "participant_id"])
y = train.answer

X_test = test.drop(columns = ["word_id", "participant_id"])

X["word_pos"] = train["word_id"].str.extract(r"_(\d+)$").astype(int)
X_test["word_pos"] = test["word_id"].str.extract(r"_(\d+)$").astype(int)

X["page"] = train["word_id"].str.extract(r"page_(\d+)_").astype(int)
X_test["page"] = test["word_id"].str.extract(r"page_(\d+)_").astype(int)

X["word"] = X["word"].apply(clean)
X_test["word"] = X_test["word"].apply(clean)

X["word_len"] = train["word"].str.len()
X_test["word_len"] = test["word"].str.len()

freq = train["word"].value_counts()
X["word_freq"] = train["word"].map(freq).fillna(0)
X_test["word_freq"] = test["word"].map(freq).fillna(0)

X["prev_word"] = train.groupby("text")["word"].shift(1)
X_test["prev_word"] = test.groupby("text")["word"].shift(1)

X["next_word"] = train.groupby("text")["word"].shift(-1)
X_test["next_word"] = test.groupby("text")["word"].shift(-1)

X = X.fillna("")
X_test = X_test.fillna("")

X.head()

,word,text,word_pos,page,word_len,word_freq,prev_word,next_word
0,luna,enc_wikimoon,0,1,4,180,,https://ro.wikipedia.org/wiki/luna
1,https://ro.wikipedia.org/wiki/luna,enc_wikimoon,1,1,34,30,luna,de
2,de,enc_wikimoon,2,1,2,6420,https://ro.wikipedia.org/wiki/luna,la
3,la,enc_wikimoon,3,1,2,1470,de,"wikipedia,"
4,"wikipedia,",enc_wikimoon,4,1,10,30,la,enciclopedia


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_char = TfidfVectorizer(analyzer = "char_wb", ngram_range = (2, 6))

X_train_word = tfidf_char.fit_transform(X_train["word"])
X_val_word = tfidf_char.transform(X_val["word"])
X_test_word = tfidf_char.transform(X_test["word"])

X_train_prev_word = tfidf_char.transform(X_train["prev_word"])
X_val_prev_word = tfidf_char.transform(X_val["prev_word"])
X_test_prev_word = tfidf_char.transform(X_test["prev_word"])

X_train_next_word = tfidf_char.transform(X_train["next_word"])
X_val_next_word = tfidf_char.transform(X_val["next_word"])
X_test_next_word = tfidf_char.transform(X_test["next_word"])

from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown="ignore")

X_train_text = ohe.fit_transform(X_train[["text"]])
X_val_text = ohe.transform(X_val[["text"]])
X_test_text = ohe.transform(X_test[["text"]])

X_train_word

<108168x23664 sparse matrix of type '<class 'numpy.float64'>'
	with 2397783 stored elements in Compressed Sparse Row format>

In [7]:
from scipy.sparse import hstack, csr_matrix

num_cols = [col for col in X.columns if X[col].dtype != 'object']

X_train_full = hstack([
    X_train_word,
    X_train_prev_word,
    X_train_next_word,
    X_train_text,
    csr_matrix(X_train[num_cols].values)
])

X_val_full = hstack([
    X_val_word,
    X_val_prev_word,
    X_val_next_word,
    X_val_text,
    csr_matrix(X_val[num_cols].values)
])

X_test_full = hstack([
    X_test_word,
    X_test_prev_word,
    X_test_next_word,
    X_test_text,
    csr_matrix(X_test[num_cols].values)
])

In [ ]:
from catboost import CatBoostRegressor
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver="saga", max_iter=50, n_jobs=-1)

model.fit(X_train_full, y_train)

preds = model.predict(X_val_full)

In [ ]:
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
import numpy as np

def eval_metric(y_true, preds):
    y_true = y_true.astype(float)
    preds = preds.astype(float)
    r2 = r2_score(y_true, preds, sample_weight=None, force_finite=True)
    r2 = max(0, r2)
    pears = pearsonr(y_true, preds)[0]
    if np.isnan(pears):
        pears = 0.0
    pears = np.abs(pears)
    return 100*(pears + r2)/2

score = eval_metric(y_val, preds)
round(score, 5)

In [ ]:
preds_full = model.predict(X_test_full)

In [ ]:
submission = pd.DataFrame({
    "subtaskID" : 1,
    "datapointID" : test.datapointID,
    "answer" : preds_full
})

submission

In [ ]:
submission.to_csv("submission.csv", index = False)